# Проект первого модуля: анализ данных для агентства недвижимости
- Автор:  Ивченко Юлия Вадимовна
- Дата:  27.11.2025

## Цели и задачи
Агентству нужен готовый анализ объявлений о продаже жилой недвижимости в Санкт-Петербурге и Ленинградской области, чтобы найти самые перспективные сегменты недвижимости. Предстоит работать с данными, в которой находится архивная информация сервиса Яндекс Недвижимость: объявления о продаже жилой недвижимости в Санкт-Петербурге и Ленинградской области за несколько лет.

## Решаем ad hoc задачи 
### Задача 1. Время активности объявлений

Чтобы спланировать эффективную бизнес-стратегию на рынке недвижимости, заказчику нужно определить — по времени активности объявления — самые привлекательные для работы сегменты недвижимости Санкт-Петербурга и городов Ленинградской области.


In [1]:

-- Определим аномальные значения (выбросы) по значению перцентилей:
WITH limits AS (
    SELECT
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats
),
-- Найдём id объявлений, которые не содержат выбросы, также оставим пропущенные данные:
filtered_id AS(
    SELECT id
    FROM real_estate.flats
    WHERE
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
    ),
categor AS(
	SELECT *,
			CASE WHEN city='Санкт-Петербург' THEN 'Санкт-Петербург'
				ELSE 'ЛенОбл'
			END AS cat_region,
			CASE WHEN days_exposition<=30 THEN 'до 1 мес'
				WHEN days_exposition<=90 THEN 'от 1 до 3 мес'
				WHEN days_exposition<=180 THEN 'от 3 до 6 мес'
				WHEN days_exposition>180 THEN 'более 6 мес'
				ELSE 'active'
			END AS cat_activ
	FROM real_estate.flats
	LEFT JOIN real_estate.city using(city_id)
	LEFT JOIN real_estate.advertisement using(id)
	LEFT JOIN real_estate.TYPE using(type_id)
	WHERE (id IN (SELECT * FROM filtered_id)) 
		AND (type='город') 
		AND (EXTRACT(YEAR FROM first_day_exposition) BETWEEN 2015 AND 2018)
			)
SELECT cat_region,
		cat_activ,
		count(id) AS count_id,
		round(count(id)/sum(count(id)) OVER (PARTITION BY cat_region),2) AS segment_id,
		round(avg(last_price/total_area::numeric)::numeric,2) AS avg_price_kv_m,
		round(avg(total_area)::NUMERIC,2) AS avg_area,
		round(percentile_disc(0.5) WITHIN GROUP (ORDER BY rooms)::numeric,2) AS mediana_rooms,
		round(percentile_disc(0.5) WITHIN GROUP (ORDER BY ceiling_height)::numeric,2) AS mediana_ceiling_height,
		round(percentile_disc(0.5) WITHIN GROUP (ORDER BY floor)::numeric,2) AS mediana_floor,
		round(percentile_disc(0.5) WITHIN GROUP (ORDER BY balcony)::numeric,2) AS mediana_balcony,
		round(percentile_disc(0.5) WITHIN GROUP (ORDER BY kitchen_area)::numeric,2) AS mediana_kitchen_area
FROM categor
GROUP BY cat_region, cat_activ
ORDER BY cat_region desc, count_id desc;


SyntaxError: invalid syntax (3450722114.py, line 1)


Проанализировав результаты, ответим на вопросы:
1. Какие категории объявлений являются самыми распространёнными в Санкт-Петербурге и городах Ленинградской области?
Самые распространенные категории в Санкт-Петербурге являются объявления с длительностью активности более 6 месяцев (3506 объявлений, доля объявлений в разрезе региона составляет 0,31) и от 1 до 3х месяцев (3020 объявлений, доля объявлений в разрезе региона составляет 0,27)
Самые распространенные категории в Ленинградской области являются объявления с длительностью активности более 6 месяцев (873 объявлений, доля объявлений в разрезе региона составляет 0,31) и от 1 до 3х месяцев (864 объявлений, доля объявлений в разрезе региона составляет 0,31)
2. Какие характеристики недвижимости, включая площадь недвижимости, среднюю стоимость квадратного метра, количество комнат и балконов и другие параметры, влияют на время активности объявлений? Как эти зависимости варьируют между регионами?
В Санкт-Петербурге средняя цена за кв.метр варьируется от 108000 до 115000 рублей. Квартиры, которые продались быстрее всего (меньше, чем за 1 мес.) стоят за кв.м 108919.78 рублей. Квартиры, которые продавались дольше всего (более 6 мес.) стоят за кв.м 114981.07. Чем дороже цена за кв.м., тем дольше продается квартира
В Ленинградской области средняя цена за кв.метр варьируется от 67000 до 72000 рублей. Самая большая средняя цена за кв.м в категории до 1 месяца - 71907.63 рублей.
Активность объявлений прямопропорционально зависит от средней площади квартиры, чем меньше квартира, тем меньше времени продается. В Санкт-Петербурге площадь от 54.66 до 65.76 кв.м. В Ленинградской области от 48.75 до 55.03 кв.м
Остальные показатели:количество комнат, высота потолков, количество балконов, этаж квартиры и площадь кухни на быстроту продаваемости практически не влияют, и примерно для каждой категории по активности объявлений одинаковы. Некоторые из этих показателей различаются только по регионам. 
3. Есть ли различия между недвижимостью Санкт-Петербурга и Ленинградской области по полученным результатам?
Да, В санкт-Петербурге цены на жилье выше, цена за кв.метр в 1,5 раза больше, чем в Ленинградской области. Площадь квартиры в Санкт-Петербурге больше, чем в Ленинградской области примерно на 10кв.м по каждой категории. Этаж в Санкт-Петербурге -5, а в Ленинградской области - 3. Площадь кухни в Санкт-Петербурге в среднем 10  кв.м, а в Ленинградской области - 8,5 кв.м. Остальные показатели одинаковы:количество комнат, высота потолков и количество балконов

### Задача 2. Сезонность объявлений

Заказчику важно понять сезонные тенденции на рынке недвижимости Санкт-Петербурга и Ленинградской области — то есть для всего региона, чтобы выявить периоды с повышенной активностью продавцов и покупателей недвижимости. Это поможет спланировать маркетинговые кампании и выбрать сроки для выхода на рынок.




In [3]:
-- Определим аномальные значения (выбросы) по значению перцентилей:
WITH limits AS (
    SELECT
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats
),
-- Найдём id объявлений, которые не содержат выбросы, также оставим пропущенные данные:
filtered_id AS(
    SELECT id
    FROM real_estate.flats
    WHERE
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
 ),
all_tabl AS(
	SELECT id,
		EXTRACT(MONTH FROM first_day_exposition::timestamp) AS first_day,
		days_exposition,
		EXTRACT(MONTH FROM first_day_exposition::timestamp+days_exposition*INTERVAL '1 day') AS last_day,
		round((last_price/total_area::NUMERIC)::numeric,2) AS price_kv_m,
		total_area
	FROM real_estate.flats
	LEFT JOIN real_estate.city using(city_id)
	LEFT JOIN real_estate.advertisement using(id)
	LEFT JOIN real_estate.TYPE using(type_id)
	WHERE (id IN (SELECT * FROM filtered_id)) 
		AND (type='город') 
		AND (EXTRACT(YEAR FROM first_day_exposition) BETWEEN 2015 AND 2018)
),
first_day_stat AS(
	SELECT first_day,
			count(id) AS count_open_advertisement,
			round(avg(price_kv_m)::NUMERIC,2) AS avg_price_kv_m_open,
			round(avg(total_area)::NUMERIC,2) AS avg_total_area_open
	FROM all_tabl
	GROUP BY first_day
),
last_day_stat AS(
	SELECT last_day,
			count(id) AS count_close_advertisement,
			round(avg(price_kv_m)::NUMERIC,2) AS avg_price_kv_m_close,
			round(avg(total_area)::NUMERIC,2) AS avg_total_area_close
	FROM all_tabl
	WHERE days_exposition IS NOT NULL 
	GROUP BY last_day
)
SELECT CASE WHEN first_day=1 THEN 'январь'
			WHEN first_day=2 THEN 'февраль'
			WHEN first_day=3 THEN 'март'
			WHEN first_day=4 THEN 'апрель'
			WHEN first_day=5 THEN 'май'
			WHEN first_day=6 THEN 'июнь'
			WHEN first_day=7 THEN 'июль'
			WHEN first_day=8 THEN 'август'
			WHEN first_day=9 THEN 'сентябрь'
			WHEN first_day=10 THEN 'октябрь'
			WHEN first_day=11 THEN 'ноябрь'
			WHEN first_day=12 THEN 'декабрь'
		END AS month_,
		count_open_advertisement,
		round(100*count_open_advertisement/sum(count_open_advertisement) OVER ()::NUMERIC,2) AS segment_open,
		avg_price_kv_m_open,
		avg_total_area_open,
		count_close_advertisement,
		round(100*count_close_advertisement/sum(count_close_advertisement) OVER ()::NUMERIC,2) AS segment_close,
		avg_price_kv_m_close,
		avg_total_area_close
FROM first_day_stat AS f
FULL JOIN last_day_stat AS l ON f.first_day=l.last_day
ORDER BY first_day;




IndentationError: unindent does not match any outer indentation level (<string>, line 66)


Проанализировав результаты, ответим на вопросы:
1. В какие месяцы наблюдается наибольшая активность в публикации объявлений о продаже недвижимости? А в какие — по снятию? Это показывает динамику активности покупателей.
Наибольшая активность в публикации объявлений о продаже недвижимости - в ноябре (1569, доля среди всех объявлений 11.17), октябре (1437, доля среди всех объявлений 10.23), феврале (1369, доля среди всех объявлений 9.75) и сентябре (1341, доля среди всех объявлений 9.55). Наименьшее количество поданных объявлений в январе (735, доля среди всех объявлений5.23) и мае (891, доля среди всех объявлений 6.34).
Наибольшее количество снятых объявлений в октябре (1360, доля среди всех объявлений 10.31), ноябре (1301, доля среди всех объявлений 9.86), сентябре (1238, доля среди всех объявлений 9.38) и январе (1225, доля среди всех объявлений 9.28). Наименьшее количество снятых объявлений в мае (729, доля среди всех объявлений 5.53) и июне (771, доля среди всех объявлений 5.84).
2. Совпадают ли периоды активной публикации объявлений и периоды, когда происходит повышенная продажа недвижимости (по месяцам снятия объявлений)?
Периоды активности подачи и снятия объявлений совпадают по месяцам:сентябрь, октябрь и ноябрь.
Хуже всего подают и снимают объявления в мае.
3. Как сезонные колебания влияют на среднюю стоимость квадратного метра и среднюю площадь квартир? Что можно сказать о зависимости этих параметров от месяца?
Сезонные колебания особо не влияют на среднюю стоимость за кв.метр, единственное, цена за метр квадратный ниже в мае и при продаче и при снятии объявления.
Сезонные колебания на среднюю площадь квартиры вообще не влияют и примерно одинаковы во всех месяцах.

### Общие выводы и рекомендации
В данных есть аномальные и нереальные значения, поэтому применили фильтрацию, чтобы данные остались корректные и аномальные значения не влияли на статистические показатели.
В Санкт-Петербурге количество объявлений намного больше, чем в Ленинградской области. Цены за метр квадратный выше в 1.5 раза, чем в области.
Активность объявлений зависит от цены за квадратный метр (чем меньше цена, тем быстрее продается квартира) и площади квартиры (чем меньше площадь, тем быстрее продается квартира). Остальные показатели практически не влияют на время продажи квартиры.
Больше всего продают и покупают квартиры в сентябре, октябре и ноябре. Сезонные колебания практически не влияют на основные показатели.
